In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.volume import OnBalanceVolumeIndicator, ChaikinMoneyFlowIndicator
from ta.volatility import AverageTrueRange
from ta.volume import VolumeWeightedAveragePrice
from scipy.optimize import minimize
from datetime import datetime
from dateutil.relativedelta import relativedelta
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb

from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
import warnings
warnings.filterwarnings('ignore')



TICKERS = [
        "PETR4.SA", "VALE3.SA", "PRIO3.SA",
        "MGLU3.SA", "LREN3.SA", "ABEV3.SA", "WEGE3.SA",
        "ELET3.SA",
        "SUZB3.SA",
        "EMBR3.SA", "RDOR3.SA", "RAIL3.SA"
    ]


TICKERS_EXPANDIDA = [
    # BANCOS (11)
    "ITUB4.SA",  # Itaú Unibanco
    "BBDC4.SA",  # Bradesco
    "BBAS3.SA",  # Banco do Brasil
    "SANB11.SA", # Santander
    "BPAC11.SA", # Banco do Brasil PN
    "CXSE3.SA",  # Caixa Seguridade
    "BRAP4.SA",  # Bradespar
    "BRSR6.SA",  # Banco do Brasil ON
    "CRFB3.SA",  # Carrefour Brasil
    "PSSA3.SA",  # Porto Seguro
    "PINE4.SA",  # Banco Pine
    
    # ENERGIA (12)
    "PETR4.SA",  # Petrobras PN
    "PRIO3.SA",  # Petrorio
    "OIBR4.SA",  # Oi PN
    "ELET3.SA",  # Eletrobras ON
    "CMIG4.SA",  # Cemig PN
    "CPFE3.SA",  # CPFL Energia
    "EGIE3.SA",  # EDP Energias
    "ENGI11.SA", # Engie Brasil
    "GEMA3.SA",  # Gerdau Metalúrgica
    "LIGHT3.SA", # Light
    "TRPL4.SA",  # Transmissão Paulista
    "EQTL3.SA",  # Equatorial Energia
    
    # MINERAÇÃO (4)
    "VALE3.SA",  # Vale
    "CSNA3.SA",  # Companhia Siderúrgica
    "USIM5.SA",  # Usiminas
    "GGBR4.SA",  # Gerdau PN
    
    # VAREJO (8)
    "MGLU3.SA",  # Magazine Luiza
    "LREN3.SA",  # Lojas Renner
    "ABEV3.SA",  # Ambev
    "RENT3.SA",  # Localiza
    "MOVI3.SA",  # Movida
    "VVAR3.SA",  # Via Varejo
    "PCAR3.SA",  # Impar
    "TRIS3.SA",  # Triscila
    
    # CONSUMO (9)
    "WEGE3.SA",  # WEG
    "JBSS3.SA",  # JBS
    "MSFT34.SA", # Microsoft (ADR)
    "HYPE3.SA",  # Hypera
    "SLCE3.SA",  # SLC Agrícola
    "PETZ3.SA",  # Petz
    "ARZZ3.SA",  # Arezzo
    "TFCO4.SA",  # Telefônico Brasil
    "BRML3.SA",  # Brasil Malha Logística
    
    # TRANSPORTE (6)
    "RAIL3.SA",  # Rumo
    "CCRO3.SA",  # CCR
    "LOGB3.SA",  # Loggi
    "ARZZ3.SA",  # Arezzo (calçados)
    "EMAE4.SA",  # Emae
    "ATUS3.SA",  # Atus
    
    # CONSTRUÇÃO (5)
    "MRVE3.SA",  # MRV Engenharia
    "TEND3.SA",  # Construtora Tenda
    "PLPL3.SA",  # Plano & Plano
    "GFSA3.SA",  # Gafisa
    "TRAD3.SA",  # Tradição
    
    # IMÓVEIS (5)
    "VLID3.SA",  # Validada Imóveis
    "BRIV3.SA",  # BR Imobiliário
    "CYRE3.SA",  # Cyrela
    "EVEN3.SA",  # Even
    "HBOR3.SA",  # Helbor
    
    # COMUNICAÇÃO (3)
    "VIVT3.SA",  # Vivo
    "TIMS3.SA",  # Tim
    "OIBR3.SA",  # Oi ON
    
    # PAPEL E CELULOSE (4)
    "SUZB3.SA",  # Suzano
    "SBSP3.SA",  # Sabesp
    "KLABIN11.SA", # Klabin
    "FIBR3.SA",  # Fibria
    
    # QUÍMICA/HIGIENE (3)
    "TOTS3.SA",  # Totvs
    "BRPR3.SA",  # Brasilfops
    "CLSA3.SA",  # Classa
    
    # ALIMENTOS (4)
    "MBLY3.SA",  # Marfrig
    "BRF3.SA",   # BRF
    "SEQL3.SA",  # Sequoia
    "ASAI3.SA",  # Assaí
    
    # TECNOLOGIA (5)
    "TOTS3.SA",  # Totvs
    "NTCO3.SA",  # Natura
    "BRQT3.SA",  # Brq Digital
    "DIRR3.SA",  # Direcional Engenharia
    "TRPL4.SA",  # Transmissão Paulista
    
    # AVIAÇÃO (3)
    "EMBR3.SA",  # Embraer
    "AZUL4.SA",  # Azul
    "GOLL4.SA",  # Gol
    
    # SEGUROS (3)
    "PSSA3.SA",  # Porto Seguro
    "SULB3.SA",  # Sulamerica
    "SGUP3.SA",  # Seguradoras Unidas
    
    # FINANCEIRAS (4)
    "B3SA3.SA",  # B3
    "MOVI3.SA",  # Movida
    "RBRR3.SA",  # Rede Brasil Real
    "RDOR3.SA",  # Rede D'Or
    
    # AGRONEGÓCIO (3)
    "AGRO3.SA",  # Agrogalaxy
        "AERI3.SA",  # Aerea Invest
    "POSI3.SA",  # Positivo
    ]


In [2]:
def baixar_e_calcular_indicadores(ticker, start="2010-01-01", end=None):
    """
    Baixa dados e calcula indicadores técnicos.
    """
    try:
        if end is not None:
            data = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        else:
            data = yf.download(ticker, start=start, progress=False, auto_adjust=True)
        
        if data.empty:
            return None
        
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.droplevel(1)
        
        serie = data['Close']
        
        # Indicadores técnicos
        data['SMA_20'] = SMAIndicator(serie, window=20).sma_indicator()
        data['SMA_50'] = SMAIndicator(serie, window=50).sma_indicator()
        data['SMA_200'] = SMAIndicator(serie, window=200).sma_indicator()
        data['EMA_12'] = EMAIndicator(serie, window=12).ema_indicator()
        data['EMA_26'] = EMAIndicator(serie, window=26).ema_indicator()
        data['EMA_50'] = EMAIndicator(serie, window=50).ema_indicator()
        data['RSI_14'] = RSIIndicator(serie, window=14).rsi()
        
        macd = MACD(serie)
        data['MACD'] = macd.macd()
        data['MACD_Hist'] = macd.macd_diff()
        data['ATR_14'] = AverageTrueRange(data['High'], data['Low'], serie, window=14).average_true_range()
        data['ROC_12'] = ROCIndicator(serie, window=12).roc()
        data['OBV'] = OnBalanceVolumeIndicator(serie, data['Volume']).on_balance_volume()
        
        # Retornos e Volatilidade
        data['Ret_1d'] = serie.pct_change(1)
        data['Ret_5d'] = serie.pct_change(5)
        data['Ret_21d'] = serie.pct_change(21)
        data['Vol_21d'] = serie.pct_change().rolling(21).std()
        data['Vol_63d'] = serie.pct_change().rolling(63).std()
        data['Ticker'] = ticker
        
        return data.fillna(method='ffill').dropna()
    except Exception as e:
        print(f"Erro ao baixar {ticker}: {e}")
        return None

def preparar_features_target(df, target_lag=21):
    """
    Prepara features defasadas e target SEM VAZAMENTO DE DADOS.
    
    CORREÇÃO CRÍTICA:
    - Calcula o target ANTES de fazer qualquer shift nas features
    - O target representa o retorno observado 21 dias à frente
    - As features são defasadas em 1 período para não usar info do dia da predição
    - Ao treinar, a data de rebalanceamento não pode ter observado esse target
    """
    data = df.copy()
    

    # 1. PRIMEIRO: Calcular o target passado (antes de qualquer shift)
    data['Close_Past'] = data['Close'].shift(target_lag)

    #print(data['Close'].tail(50), data['Close_Past'].tail(50))

    data['Target'] = (data['Close'] - data['Close_Past']) / data['Close_Past']

    data['Target_21'] = data['Target'].shift(21)
    data['Target_42'] = data['Target'].shift(42)
    #data['Target'] = data['Close'].pct_change(periods=target_lag)
    # 2. DEPOIS: Defasar as features em 1 período
    feature_cols = ['SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'EMA_50',
                    'RSI_14', 'MACD', 'MACD_Hist', 'ATR_14', 'ROC_12', 'OBV',
                    'Ret_1d', 'Ret_5d', 'Ret_21d', 'Vol_21d', 'Vol_63d','Target_21','Target_42']
    
    for col in feature_cols:
        data[f'{col}_lag1'] = data[col].shift(1)
    
    # 3. REMOVER últimos target_lag linhas (pois não têm target válido)
    data = data.iloc[:-target_lag].copy()
    
    # 4. Usar apenas features defasadas
    feature_cols_lag = [col for col in data.columns if col.endswith('_lag1')]
    data = data.dropna(subset=['Target'] + feature_cols_lag).copy()
    

    print(data[['Close', 'Target','Close_Past']].tail(50))

    return data, feature_cols_lag

'''
def treinar_e_prever(df, feature_cols_lag, data_predicao, ticker, modelo_tipo='RF', target_lag=1):
    """
    Treina modelo SEM VAZAMENTO DE DADOS - CORRIGIDO COM LÓGICA DE PREDIÇÃO.
    
    LÓGICA CRÍTICA:
    - Se queremos PREVER em data_predicao, e temos lag de 21 dias
    - O target de cada linha = retorno 21 dias no FUTURO
    - Portanto, para treinar SEM VAZAMENTO:
      * Treinar APENAS até (data_predicao - target_lag)
      * Isso garante que o target da última linha será observado apenas APÓS data_predicao
    
    - PARA FAZER A PREDIÇÃO:
      * Queremos features do dia data_predicao (ou próximo dia útil)
      * Mas essas features NÃO FORAM CALCULADAS NOS DADOS DE TREINO
      * Precisamos buscar NO DATAFRAME ORIGINAL as features para data_predicao
      * E passar para o modelo treinado
    
    Exemplo:
    - Queremos prever retorno em 31/08/2025
    - Treinar até: 31/08 - 21 = 10/08 (última linha com target conhecido)
    - Predição: usar features de 31/08 (primeira data >= data_predicao)
    - Resultado: previsão para próximos 21 dias (até 21/09)
    """
    # Data limite de treino: (data_predicao - target_lag)
    data_limite_treino = data_predicao - pd.Timedelta(days=target_lag)
    
    # Treinar APENAS com dados ANTES dessa data
    df_treino = df[df.index <= data_limite_treino].copy()
    
    if len(df_treino) < 252:
        print(f" ❌ Dados insuficientes: {len(df_treino)} dias")
        return None
    
    # Extrair X e y para treino
    X_train = df_treino[feature_cols_lag]
    y_train = df_treino['Target']
    
    # Escolher modelo
    if modelo_tipo == 'RF':
        model = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        )
    else:  # OLS
        model = LinearRegression()
    
    model.fit(X_train, y_train)
    
    # ✅ CORREÇÃO: Buscar features para data_predicao no DF ORIGINAL (não no de treino)
    # Encontrar primeira data >= data_predicao
    df_futuro = df[df.index >= data_predicao]
    
    if len(df_futuro) == 0:
        print(f" ❌ Sem dados disponíveis para predição em/após {data_predicao.strftime('%Y-%m-%d')}")
        return None
    
    linha_predicao = df_futuro.iloc[[0]]  # Primeira linha >= data_predicao
    data_real_predicao = linha_predicao.index[0]
    
    # Verificar se features têm NaN
    features_com_nan = linha_predicao[feature_cols_lag].isna().sum().sum()
    if features_com_nan > 0:
        print(f" ⚠ AVISO: {features_com_nan} features com NaN na linha de predição")
    
    # Fazer predição com linha de data_predicao
    X_pred = linha_predicao[feature_cols_lag]
    predicao = model.predict(X_pred)[0]
    
    if predicao > 0:
        print(f"\n{'─'*70}")
        print(f"🔍 MODELO: {modelo_tipo} - {ticker}")
        print(f"📊 DADOS DE TREINO:")
        print(f" • Início: {df_treino.index[0].strftime('%Y-%m-%d')}")
        print(f" • Fim: {df_treino.index[-1].strftime('%Y-%m-%d')} ← ÚLTIMA LINHA COM TARGET CONHECIDO")
        print(f"📊 DADOS DE PREDIÇÃO:")
        print(f" • Data limite treino: {data_limite_treino.strftime('%Y-%m-%d')} (= data_predicao - {target_lag}d)")
        print(f" • Data solicitada: {data_predicao.strftime('%Y-%m-%d')}")
        print(f" • Data real usada: {data_real_predicao.strftime('%Y-%m-%d')} ← FEATURES PARA PREDIÇÃO")
        print(f" • Total dias treino: {len(df_treino)} dias")
        print(f" • Preço em {data_real_predicao.strftime('%Y-%m-%d')}: R$ {linha_predicao['Close'].iloc[0]:.2f}")
        print(f" ✅ Predição: {predicao*100:+.2f}% (para próximos {target_lag} dias)")
    
    return predicao

'''
def calcular_matriz_covariancia(tickers_list, dados_acoes, data_ref, janela=63, target_lag=21):
    """
    Calcula matriz de covariância APENAS com dados históricos.
    
    CORREÇÃO: data_ref deve ser ajustada para (data_ref - target_lag)
    para evitar vazamento temporal
    """
    retornos_hist = []
    
    # Ajustar data de referência: usar dados até (data_ref - target_lag)
    data_limite = data_ref - pd.Timedelta(days=target_lag)
    
    print(f"\nDatas da matriz de covariancia (ajustada para lag={target_lag})")
    print(f"Data referência: {data_ref.strftime('%Y-%m-%d')} → Data limite: {data_limite.strftime('%Y-%m-%d')}")
    
    for ticker in tickers_list:
        df = dados_acoes[ticker]
        # Usar dados até data_limite (não data_ref)
        df_periodo = df[df.index <= data_limite].tail(janela)
        print(f"• {ticker}: {df_periodo.index[0].strftime('%Y-%m-%d')} a {df_periodo.index[-1].strftime('%Y-%m-%d')}")
        
        if len(df_periodo) >= 21:
            retornos = df_periodo['Close'].pct_change().dropna()
            retornos_hist.append(retornos.values)
        else:
            retornos_hist.append(np.zeros(janela-1))
    
    min_len = min(len(r) for r in retornos_hist)
    retornos_df = pd.DataFrame({ticker: r[-min_len:] for ticker, r in zip(tickers_list, retornos_hist)})
    cov_matrix = retornos_df.cov()
    
    return cov_matrix



def otimizar_markowitz(retornos_esperados, cov_matrix, risk_free_rate=0.10/252):
    """
    Otimização de Markowitz: maximizar Sharpe Ratio.
    """
    tickers = list(retornos_esperados.keys())
    n_ativos = len(tickers)
    mu = np.array([retornos_esperados[t] for t in tickers])
    cov = cov_matrix.loc[tickers, tickers].values
    
    def negative_sharpe(weights):
        portfolio_return = np.dot(weights, mu)
        portfolio_vol = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))
        sharpe = (portfolio_return - risk_free_rate) / portfolio_vol if portfolio_vol > 0 else 0
        return -sharpe
    
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = tuple((0, 0.4) for _ in range(n_ativos))
    w0 = np.array([1/n_ativos] * n_ativos)
    
    result = minimize(
        negative_sharpe,
        w0,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
        options={'maxiter': 1000}
    )
    
    if result.success:
        pesos = {ticker: peso for ticker, peso in zip(tickers, result.x) if peso > 0.001}
    else:
        pesos = {ticker: 1/n_ativos for ticker in tickers}
    
    return pesos

def calcular_max_drawdown(retornos):
    """Calcula o máximo drawdown."""
    valor_acumulado = (1 + pd.Series(retornos)).cumprod()
    max_anterior = valor_acumulado.cummax()
    drawdown = (valor_acumulado - max_anterior) / max_anterior
    return drawdown.min()
    

In [3]:
def random_forest(df, feature_cols_lag, data_predicao, ticker, modelo_tipo='RF', 
                               target_lag=21, usar_validacao=True, k=15):


    data_limite_treino = data_predicao - pd.Timedelta(days=target_lag)
    
    # Treinar APENAS com dados ANTES dessa data
    df_treino = df[df.index <= data_predicao].copy()
    
    df_treino['Target'] = df_treino['Target'].shift(target_lag)

    df_treino = df_treino[df_treino.index <= data_limite_treino].copy()

    if len(df_treino) < 252:
        print(f" ❌ Dados insuficientes: {len(df_treino)} dias")
        return None
    
    
    features_selecionadas, selector = selecionar_features_importantes(
        df_treino, 
        feature_cols_lag, 
        k=min(k, len(feature_cols_lag)//2)
    )
    
    if len(features_selecionadas) < 3:
        print(f" ⚠ Poucas features selecionadas")
        return None
    
    df_treino = df_treino.dropna(subset=features_selecionadas + ['Target'])
    scaler = StandardScaler()
    X_train = scaler.fit_transform(df_treino[features_selecionadas])
    y_train = df_treino['Target'].values
    
    # ✅ MELHORIA 3: Validação cruzada temporal
    if usar_validacao and len(df_treino) > 500:
        #print(f"   Validando com Time Series Split...")
        tscv = TimeSeriesSplit(n_splits=3)
        scores = []
        
        for train_idx, val_idx in tscv.split(X_train):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
            if True: #modelo_tipo == 'RF':
                model_temp = RandomForestRegressor(
                    n_estimators=150,
                    max_depth=8,
                    min_samples_split=30,
                    min_samples_leaf=15,
                    max_features='sqrt',
                    random_state=42,
                    n_jobs=-1
                )

            else:
                model_temp = LinearRegression()
            
            model_temp.fit(X_tr, y_tr)
            score = model_temp.score(X_val, y_val)
            scores.append(score)
        
        cv_score = np.mean(scores)
        #print(f"   CV R² Score: {cv_score:.4f}")
    
    # ✅ MELHORIA 4: Hiperparâmetros otimizados para Random Forest
    if modelo_tipo == 'RF':
        model = RandomForestRegressor(
            n_estimators=214,           # ↑ Mais árvores
            max_depth=21,                # ↓ Reduz overfitting
            min_samples_split=30,       # ↑ Mais restritivo
            min_samples_leaf=9,        # ↑ Mais restritivo
            max_features='sqrt',        # Reduz correlação entre árvores
            min_weight_fraction_leaf=0.02,  # Reduz influence de outliers
            random_state=42,
            n_jobs=-1
        )
    #{'max_depth': 21, 'max_features': 'sqrt', 'min_samples_leaf': 9, 'n_estimators': 214}
    else:
        model = LinearRegression()
    
    model.fit(X_train, y_train)
    
    # ✅ MELHORIA 5: Calcular importância de features
    if modelo_tipo == 'RF':
        feature_importance = model.feature_importances_
        top_features_idx = np.argsort(feature_importance)[-5:]
        #print(f"   Top 5 features mais importantes:")
        #for idx in sorted(top_features_idx, reverse=True):
            #print(f"   • {features_selecionadas[idx]}: {feature_importance[idx]:.4f}")
    
    # Buscar features para data_predicao
    df_futuro = df[df.index >= data_predicao]
    
    if len(df_futuro) == 0:
        print(f" ❌ Sem dados disponíveis para predição em/após {data_predicao.strftime('%Y-%m-%d')}")
        return None
    
    linha_predicao = df_futuro.iloc[[0]]
    data_real_predicao = linha_predicao.index[0]
    
    # ✅ Normalizar features de predição com mesma escala
    X_pred = scaler.transform(linha_predicao[features_selecionadas])
    predicao = model.predict(X_pred)[0]
    
    #print(f"📊 DADOS DE TREINO:")
    #print(f" • Início: {df_treino.index[0].strftime('%Y-%m-%d')}")
    #print(f" • Fim: {df_treino.index[-1].strftime('%Y-%m-%d')}")
    #print(f" • Total dias: {len(df_treino)} dias")
    #print(f" • Data solicitada: {data_predicao.strftime('%Y-%m-%d')}")
    #print(f" • Data usada: {data_real_predicao.strftime('%Y-%m-%d')}")
    #print(f" • Preço: R$ {linha_predicao['Close'].iloc[0]:.2f}")
    #print(f" ✅ Predição: {predicao*100:+.2f}% (próximos {target_lag} dias)")
    
    return predicao


In [ ]:
def selecionar_features_importantes(df, feature_cols_lag, target_col='Target', k=5):
    """
    Seleciona apenas as k features mais importantes via SelectKBest.
    Reduz overfitting e ruído.
    """
    df_ = df.copy()
    df_clean = df_.dropna(subset=feature_cols_lag + [target_col])
    X = df_clean[feature_cols_lag]
    y = df_clean[target_col]
    
    selector = SelectKBest(score_func=f_regression, k=min(k, len(feature_cols_lag)))
    X_selected = selector.fit_transform(X, y)
    
    # Obter nomes das features selecionadas
    selected_mask = selector.get_support()
    features_selecionadas = [col for col, selected in zip(feature_cols_lag, selected_mask) if selected]
    

    
    return features_selecionadas, selector

def treinar_e_prever(df, feature_cols_lag, data_predicao, ticker, modelo_tipo='RF', 
                               target_lag=21, usar_validacao=True, k=15):
    """
    Treina modelo COM MELHORIAS:
    - Seleção de features (evita ruído)
    - Normalização de features
    - Hiperparâmetros otimizados para RF
    - Validação cruzada temporal
    - Regularização
    """

    if modelo_tipo == 'media3m':
        prev = previsao_media(df, data_predicao,quantidade_de_meses = 3,vetor_pesos = [5,3,2])
        previsao = min(prev,0.05)
    elif modelo_tipo == 'RF':
        previsao = random_forest(df, feature_cols_lag, data_predicao, ticker)

    elif modelo_tipo == 'OLS':
        print(f" ❌ Modelo OLS não implementado")
    
    elif modelo_tipo == 'menorvariancia':
        previsao = 0

    elif modelo_tipo == 'retornohistorico':
        previsao = df['Target'][df.index < data_predicao].mean()

    elif modelo_tipo == 'retornoultimos5m':
        previsao = previsao_media(df, data_predicao)

    elif modelo_tipo == 'menosretornoultimos5m':
        previsao = -previsao_media(df, data_predicao)

    elif modelo_tipo == 'retornoultimomes':
        previsao = previsao_media(df, data_predicao,quantidade_de_meses = 1,vetor_pesos = [1])

    if previsao is None:
        print(f"[{ticker}] Não foi possível calcular previsão (3M) na data {data_predicao}.")
        return None
    print(f"📊 PREDIÇÃO | {ticker}: {previsao}\n")

    return previsao


def preparar_features_target(df, target_lag=21):
    """
    Versão melhorada que cria features mais robustas:
    - Adiciona relações entre indicadores
    - Remove features altamente correlacionadas
    - Adiciona volatilidade condicional
    """
    data = df.copy()
    
    # 1. Calcular target
    data['Close_Future'] = data['Close'].shift(-target_lag)
    data['Target'] = (data['Close_Future'] - data['Close']) / data['Close']
    
    # 2. Features base
    feature_cols = ['SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'EMA_50',
                    'RSI_14', 'MACD', 'MACD_Hist', 'ATR_14', 'ROC_12', 'OBV',
                    'Ret_1d', 'Ret_5d', 'Ret_21d', 'Vol_21d', 'Vol_63d']
    
    # 3. ✅ NOVO: Features derivadas (relações)
    # Preço vs média móvel (momentum)
    data['Price_SMA20_Ratio'] = data['Close'] / data['SMA_20'] - 1
    data['Price_SMA50_Ratio'] = data['Close'] / data['SMA_50'] - 1
    
    # Volatilidade relativa
    data['Vol_Ratio_21_63'] = data['Vol_21d'] / (data['Vol_63d'] + 1e-8)
    
    # Força do trend (EMA cruzamentos)
    data['EMA_Crossover_12_26'] = data['EMA_12'] - data['EMA_26']
    
    # RSI momentum
    data['RSI_Velocity'] = data['RSI_14'].diff(5)
    
    feature_cols.extend(['Price_SMA20_Ratio', 'Price_SMA50_Ratio', 'Vol_Ratio_21_63', 
                         'EMA_Crossover_12_26', 'RSI_Velocity'])
    
    # 4. Defasar features
    for col in feature_cols:
        data[f'{col}_lag1'] = data[col].shift(1)
    
    # 5. Remover últimos target_lag
    data = data.iloc[:-target_lag].copy()
    
    # 6. Features defasadas
    feature_cols_lag = [col for col in data.columns if col.endswith('_lag1')]
    data = data.dropna(subset=['Target'] + feature_cols_lag).copy()
    
    return data, feature_cols_lag


In [5]:
def previsao_media(df, data_predicao, quantidade_de_meses = 5,vetor_pesos = [1,1,1,1,1], col_retorno='Target'):

    df_filtrado = df[df.index < data_predicao]
    df_filtrado['Target'] = df_filtrado['Target'].shift(1)

    if df_filtrado.empty:
        print("❌ Não há dados anteriores à data de previsão.")
        return None
    
    #print(df_filtrado[['Target','Close']].tail(30))
    media = []
    
    for i in range(quantidade_de_meses):
        peso = vetor_pesos[i]/sum(vetor_pesos)
        media.append(df_filtrado.iloc[-(i*21+21)]['Target']*peso)

    prev = np.array(media).mean()
    try:
        return prev
    except:
        print(f"Não foi possivel pegar os dados para os dias anteriores a {data_predicao}")
        return 0
'''
def previsao_media_ponderada_3m(df, data_predicao, col_retorno='Target'):


    df_filtrado = df[df.index < data_predicao]
    #print(f"Vamos conferir se isso esta certo\nData de predicao: {data_predicao.index()}\nData do df {df_filtrado.index()}\n\n\nData de ontem{df_filtrado.iloc[-1].index()}")
    if df_filtrado.empty:
        print("❌ Não há dados anteriores à data de previsão.")
        return None
    
    try:
        prev = np.array([df_filtrado.iloc[-1]['Target']*0.7,
                        df_filtrado.iloc[-21]['Target']*0.2,
                        df_filtrado.iloc[-42]['Target']*0.1]).sum()
        rsi = df_filtrado.iloc[-1]['RSI_14']
        if rsi > 90:
            fator_reducao = 1 - ((rsi_atual - 90) / 10)  # varia de 1.0 a 0.0
            fator_reducao = max(0.2, fator_reducao)  # mínimo de 20%
            prev *= fator_reducao
        return prev
    except:
        print(f"Não foi possivel pegar os dados para os dias anteriores a {data_predicao}")
        
        return 0
'''
'''
def previsao_media_5m(df, data_predicao, col_retorno='Target'):

    df_filtrado = df[df.index < data_predicao]

    if df_filtrado.empty:
        print("❌ Não há dados anteriores à data de previsão.")
        return None
    
    try:
        prev = np.array([df_filtrado.iloc[-1]['Target'],
                        df_filtrado.iloc[-21]['Target'],
                        df_filtrado.iloc[-42]['Target'],
                        df_filtrado.iloc[-63]['Target'],
                        df_filtrado.iloc[-84]['Target']]).mean()


        rsi = df_filtrado.iloc[-1]['RSI_14']
        if rsi > 90:
            fator_reducao = 1 - ((rsi_atual - 90) / 10)  # varia de 1.0 a 0.0
            fator_reducao = max(0.2, fator_reducao)  # mínimo de 20%
            prev *= fator_reducao
        return prev
    except:
        print(f"Não foi possivel pegar os dados para os dias anteriores a {data_predicao}")
        return 0
'''

'''
def previsao_media_5m(df, data_predicao, col_retorno='Target', rsi_col='RSI_14'):
    """
    Previsão = média dos retornos mensais dos últimos 5 meses (anteriores à data_predicao).
    - df: DataFrame com índice datetime ordenado ascendentemente e colunas col_retorno (diário).
    - data_predicao: pd.Timestamp ou string conversível para datetime. PREVISÃO NÃO DEVE INCLUIR data_predicao.
    - Retorna: média dos últimos 5 retornos mensais (float) ou None se dados insuficientes.
    """
    # garantir datetime e ordenação
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("O índice do df precisa ser DatetimeIndex.")
    df = df.sort_index()

    data_predicao = pd.to_datetime(data_predicao)

    # usar somente dados estritamente anteriores à data_predicao -> evita vazamento
    df_anterior = df[df.index < data_predicao].copy()
    if df_anterior.empty:
        # sem dados anteriores: não tem como prever
        return None

    # 1) construir série mensal (último preço útil do mês), usando apenas dados anteriores
    monthly_prices = df_anterior[col_retorno].resample("M").last().dropna()

    # precisa de pelo menos 2 meses para calcular um retorno mensal
    monthly_rets = monthly_prices.pct_change().dropna()
    if monthly_rets.empty:
        return None

    # 2) pegar média dos últimos 5 meses (ou menos, se não houver 5)
    last_n = 5
    last_rets = monthly_rets.tail(last_n)
    if len(last_rets) == 0:
        return None
    previsao = last_rets.mean()

    # 3) ajuste opcional por RSI: pegar RSI calculado até o último dia disponível (antes da data_predicao)
    #    Aqui assumimos que o RSI foi calculado previamente no df (coluna rsi_col).
    if rsi_col in df_anterior.columns:
        rsi_atual = df_anterior[rsi_col].iloc[-1]  # RSI do último dia útil anterior à previsão
        # Exemplo de ajuste: penaliza a previsão se RSI muito alto (sobrecompra)
        if rsi_atual > 90:
            fator_reducao = 1 - ((rsi_atual - 90) / 10)  # 90->1.0 ; 100->0.0
            fator_reducao = max(0.2, fator_reducao)     # floor em 0.2
            previsao = previsao * fator_reducao

    return float(previsao)
'''

'\ndef previsao_media_5m(df, data_predicao, col_retorno=\'Target\', rsi_col=\'RSI_14\'):\n    """\n    Previsão = média dos retornos mensais dos últimos 5 meses (anteriores à data_predicao).\n    - df: DataFrame com índice datetime ordenado ascendentemente e colunas col_retorno (diário).\n    - data_predicao: pd.Timestamp ou string conversível para datetime. PREVISÃO NÃO DEVE INCLUIR data_predicao.\n    - Retorna: média dos últimos 5 retornos mensais (float) ou None se dados insuficientes.\n    """\n    # garantir datetime e ordenação\n    if not isinstance(df.index, pd.DatetimeIndex):\n        raise ValueError("O índice do df precisa ser DatetimeIndex.")\n    df = df.sort_index()\n\n    data_predicao = pd.to_datetime(data_predicao)\n\n    # usar somente dados estritamente anteriores à data_predicao -> evita vazamento\n    df_anterior = df[df.index < data_predicao].copy()\n    if df_anterior.empty:\n        # sem dados anteriores: não tem como prever\n        return None\n\n    # 1) co

In [6]:
def calcular_fator_confianca(historico):
    """
    Retorna um número entre 1.0 (perfeito) e 3.0 (péssimo)
    que representa quanto AUMENTAR o risco percebido
    """
    if len(historico['previsto']) < 2:
        return 2.0  # Sem histórico = confiança média
    
    previsto = np.array(historico['previsto'])
    real = np.array(historico['real'])
    
    # Erro médio absoluto
    mae = np.mean(np.abs(real - previsto))
    
    # Converter MAE em fator de penalização
    # MAE = 0% → fator = 1.0 (sem ajuste)
    # MAE = 5% → fator = 2.0 (dobra o risco)
    # MAE = 10%+ → fator = 3.0 (triplica o risco)
    fator = 1.0 + min(mae * 20, 2.0)  # Limita em 3.0
    
    return fator

In [7]:
def calcular_matriz_covariancia_ajustada(tickers_list, dados_acoes, 
                                          historico_predicoes,
                                          data_ref, janela=63, target_lag=21):
    """
    IGUAL à função original, MAS ajusta pelo histórico
    """
    data_limite = data_ref - pd.Timedelta(days=target_lag)
    
    retornos_hist = []
    fatores_ajuste = []  # ✅ NOVO
    
    print(f"\n📊 Ajustando matriz de covariância por confiança:")
    
    for ticker in tickers_list:
        df = dados_acoes[ticker]
        df_periodo = df[df.index <= data_limite].tail(janela)
        
        if len(df_periodo) >= 21:
            retornos = df_periodo['Close'].pct_change().dropna()
            retornos_hist.append(retornos.values)
            
            # ✅ NOVO: Calcular fator de ajuste
            if ticker in historico_predicoes and len(historico_predicoes[ticker]['previsto']) > 0:
                fator = calcular_fator_confianca(historico_predicoes[ticker])
            else:
                fator = 2.0  # Sem histórico = confiança média
            
            fatores_ajuste.append(fator)
            print(f"  {ticker}: fator = {fator:.2f}x")
        else:
            retornos_hist.append(np.zeros(janela-1))
            fatores_ajuste.append(2.0)
    
    # Criar DataFrame (igual ao original)
    min_len = min(len(r) for r in retornos_hist)
    retornos_df = pd.DataFrame({
        ticker: r[-min_len:] for ticker, r in zip(tickers_list, retornos_hist)
    })
    
    # Covariância base (igual ao original)
    cov_matrix = retornos_df.cov()
    
    # ✅ NOVO: Ajustar a diagonal (volatilidade individual)
    for i, ticker in enumerate(tickers_list):
        cov_matrix.loc[ticker, ticker] *= fatores_ajuste[i]
    
    return cov_matrix

In [8]:
def estrategia_portfolio_mensal(tickers, data_inicio_teste="2024-01-01",
                               data_fim_teste="2024-12-31", n_acoes=8,
                               modelo_tipo='RF',risco_adaptado = True, salvar_csv=True, target_lag=21, k=15):
    """
    Estratégia de portfólio mensal com otimização de Markowitz.
    
    CORREÇÃO FINAL: Lógica de datas consertada
    """
    target_lag = target_lag
    
    data_inicio_teste = pd.to_datetime(data_inicio_teste)
    data_fim_teste = pd.to_datetime(data_fim_teste)
    
    # Baixar dados até alguns dias APÓS data_fim_teste para ter margem
    data_inicio_download = data_inicio_teste - pd.DateOffset(years=5)
    data_fim_download = data_fim_teste + pd.DateOffset(days=30)
    
    print(f"\n{'='*80}")
    print(f"ESTRATÉGIA DE PORTFÓLIO MENSAL - MODELO: {modelo_tipo}")
    print(f"{'='*80}")
    print(f"Período de teste: {data_inicio_teste.strftime('%Y-%m-%d')} até {data_fim_teste.strftime('%Y-%m-%d')}")
    print(f"Ações no universo: {n_acoes}")
    print(f"Total de tickers: {len(tickers)}")
    print(f"Target lag: {target_lag} dias")
    print(f"{'='*80}\n")
    
    print(f"📥 Baixando dados históricos...")
    print(f" • Período: {data_inicio_download.strftime('%Y-%m-%d')} até {data_fim_download.strftime('%Y-%m-%d')}")
    
    dados_acoes = {}
    for ticker in tickers:
        print(f" → {ticker}", end="... ")
        df = baixar_e_calcular_indicadores(
            ticker,
            start=data_inicio_download.strftime('%Y-%m-%d'),
            end=data_fim_download.strftime('%Y-%m-%d')
        )
        if df is not None and len(df) > 252:
            dados_acoes[ticker] = df
            print(f"✓ ({len(df)} dias, até {df.index[-1].strftime('%Y-%m-%d')})")
        else:
            print("✗ (sem dados suficientes)")
    
    print(f"\n✓ Dados baixados para {len(dados_acoes)} ações")
    
    if len(dados_acoes) < n_acoes:
        n_acoes = len(dados_acoes)
    

    dados_preparados = {}
    for ticker, df in dados_acoes.items():
        print(f"Preparando {ticker}...")
            
        df_prep, feature_cols = preparar_features_target(df, target_lag=21)
        if len(df_prep) > 0:
            dados_preparados[ticker] = (df_prep, feature_cols)





    
    print(f"✓ Features preparadas para {len(dados_preparados)} ações\n")
    
    # ✅ CORREÇÃO: Usar MS (month start) para primeira data, depois adicionar mês manualmente
    # Isso garante que as datas de rebalanceamento correspondam aos primeiros dias dos meses
    datas_rebalanceamento = pd.date_range(
        start=data_inicio_teste,
        end=data_fim_teste,
        freq='MS'
    )
    
    print(f"📅 Datas de rebalanceamento geradas: {len(datas_rebalanceamento)} meses")
    for i, d in enumerate(datas_rebalanceamento):
        print(f"   {i+1}. {d.strftime('%Y-%m-%d')}")
    print()
    
    resultados_mensais = []
    portfolios_mensais = []
    historico_predicoes = {}
    # Para cada mês
    for i, data_rebal_original in enumerate(datas_rebalanceamento):
        print(f"\n{'─'*80}")
        print(f"MÊS {i+1}: {data_rebal_original.strftime('%B/%Y').upper()}")
        print(f"{'─'*80}")
        
        # ✅ CORREÇÃO: Encontrar primeiro dia útil com dados a partir de data_rebal_original
        data_rebal = None
        offset_usado = 0
        
        for offset in range(10):
            data_teste = data_rebal_original + pd.Timedelta(days=offset)
            # Verificar se há dados disponíveis para esta data
            tem_dados = any(
                len(df[df.index >= data_teste]) > 0
                for df, _ in dados_preparados.values()
            )
            if tem_dados:
                data_rebal = data_teste
                offset_usado = offset
                break
        
        if data_rebal is None:
            print(f"⚠ Sem data válida para rebalanceamento")
            continue
        
        if offset_usado > 0:
            print(f"📅 Data ajustada: {data_rebal.strftime('%Y-%m-%d')} (+{offset_usado} dias)")
        
        # ✅ CORREÇÃO: Data fim do mês = início do próximo mês - 1 dia
        if i < len(datas_rebalanceamento) - 1:
            # Próximo mês começa em datas_rebalanceamento[i+1]
            # Então fim deste mês é um dia antes
            data_fim_mes = datas_rebalanceamento[i+1] - pd.Timedelta(days=1)
        else:
            # Último mês: usar data_fim_teste ou última data disponível, o que for menor
            ultima_data_disponivel = min([df.index[-1] for df, _ in dados_preparados.values()])
            data_fim_mes = min(data_fim_teste, ultima_data_disponivel)
        
        print(f"📅 Período do mês: {data_rebal.strftime('%Y-%m-%d')} até {data_fim_mes.strftime('%Y-%m-%d')}")
        
        # Verificar se há dias úteis suficientes após data_rebal
        dias_disponiveis = []
        for df, _ in dados_preparados.values():
            dias_apos = len(df[(df.index > data_rebal) & (df.index <= data_fim_mes)])
            dias_disponiveis.append(dias_apos)
        
        if max(dias_disponiveis) < 2:
            print(f"⚠ Dias insuficientes após {data_rebal.strftime('%Y-%m-%d')} (máx: {max(dias_disponiveis)} dias)")
            continue

        predicoes = {}
        for ticker, (df, feature_cols) in dados_preparados.items():
            pred = treinar_e_prever(df, feature_cols, data_rebal, ticker, 
                                    modelo_tipo=modelo_tipo, target_lag=21, k=k)
            if pred is not None:
                predicoes[ticker] = pred

        print(f"\n✓ Predições realizadas: {len(predicoes)} ações")
        
        if len(predicoes) < 3:
            print(f"⚠ Poucas predições, pulando")
            continue
        
        # Selecionar top N
        predicoes_sorted = sorted(predicoes.items(), key=lambda x: x[1], reverse=True)
        top_acoes = predicoes_sorted[:min(n_acoes, len(predicoes_sorted))]
        
        print(f"\n📊 TOP {len(top_acoes)} AÇÕES:")
        for rank, (ticker, ret_pred) in enumerate(top_acoes, 1):
            print(f" {rank}. {ticker:12s} → Retorno previsto: {ret_pred*100:+.2f}%")
        ###########################################################################
        # Covariância
        tickers_top = [t for t, _ in top_acoes]
        retornos_esperados = {t: r for t, r in top_acoes}

        if risco_adaptado:
            if i == 0:
                # Primeiro mês: sem histórico ainda, usar função original
                cov_matrix = calcular_matriz_covariancia(
                    tickers_top, dados_acoes, data_rebal, 
                    janela=63, target_lag=target_lag
                )
            else:
                # A partir do segundo mês: usar função ajustada
                cov_matrix = calcular_matriz_covariancia_ajustada(
                    tickers_top, dados_acoes, historico_predicoes,
                    data_rebal, janela=63, target_lag=target_lag
                )
        else:
            cov_matrix = calcular_matriz_covariancia(
                tickers_top, dados_acoes, data_rebal, 
                janela=63, target_lag=target_lag
            )
        
        # Resto continua igual (otimizar_markowitz, etc.)
        pesos_otimos = otimizar_markowitz(retornos_esperados, cov_matrix)

        
        print(f"\n🎯 PESOS OTIMIZADOS:")
        for ticker, peso in sorted(pesos_otimos.items(), key=lambda x: x[1], reverse=True):
            print(f" {ticker:12s} → {peso*100:5.2f}%")
        #############################################################################################
        # Salvar portfólio do mês
        portfolio_mes = {
            'Mes': data_rebal_original.strftime('%Y-%m'),
            'Data': data_rebal_original.strftime('%Y-%m-%d'),
            'Modelo': modelo_tipo
        }
        for ticker, peso in pesos_otimos.items():
            portfolio_mes[ticker] = peso
        portfolios_mensais.append(portfolio_mes)
        
        # Calcular retornos reais
        print(f"\n🔍 Calculando retornos reais DE {data_rebal.strftime('%Y-%m-%d')} ATÉ {data_fim_mes.strftime('%Y-%m-%d')}")
        
        retornos_reais = {}
        print(f"\n📈 RETORNOS REAIS:")
        for ticker in pesos_otimos.keys():
            df_acao = dados_acoes[ticker]
            df_periodo = df_acao[(df_acao.index > data_rebal) & (df_acao.index <= data_fim_mes)]
            
            if len(df_periodo) >= 2:
                preco_inicio = df_periodo['Close'].iloc[0]
                preco_fim = df_periodo['Close'].iloc[-1]
                retorno_real = (preco_fim - preco_inicio) / preco_inicio
                retornos_reais[ticker] = retorno_real
                
                ret_previsto = retornos_esperados[ticker]
                erro = abs(retorno_real - ret_previsto)
                print(f" {ticker:12s}: Real={retorno_real*100:+6.2f}% | Previsto={ret_previsto*100:+6.2f}% | Erro={erro*100:6.2f}%")
            else:
                retornos_reais[ticker] = 0
            # Se fizemos previsão E temos retorno real para esta ação
            if ticker in predicoes and ticker in retornos_reais:
                
                # Criar entrada se não existe
                if ticker not in historico_predicoes:
                    historico_predicoes[ticker] = {
                        'previsto': [],
                        'real': []
                    }
                
                # Adicionar este mês
                historico_predicoes[ticker]['previsto'].append(predicoes[ticker])
                historico_predicoes[ticker]['real'].append(retornos_reais[ticker])
        
        # Retorno do portfólio
        retorno_portfolio = sum(retornos_reais.get(t, 0) * peso for t, peso in pesos_otimos.items())

        # Benchmark
        ibov = yf.download("^BVSP", start=data_rebal, end=data_fim_mes, progress=False)
        if not ibov.empty and len(ibov) >= 2:
            if isinstance(ibov['Close'], pd.Series):
                preco_ibov_inicio = ibov['Close'].iloc[0]
                preco_ibov_fim = ibov['Close'].iloc[-1]
            else:
                preco_ibov_inicio = ibov['Close'].iloc[0].values[0]
                preco_ibov_fim = ibov['Close'].iloc[-1].values[0]
            retorno_ibov = (preco_ibov_fim - preco_ibov_inicio) / preco_ibov_inicio
        else:
            retorno_ibov = 0
        
        alpha = retorno_portfolio - retorno_ibov
        
        print(f"\n💰 RESULTADO DO MÊS:")
        print(f" Retorno do Portfólio: {retorno_portfolio*100:+.2f}%")
        print(f" Retorno do Ibovespa: {retorno_ibov*100:+.2f}%")
        print(f" Alpha: {alpha*100:+.2f}%")
        
        portfolio_str = ', '.join([f"{t} ({p*100:.1f}%)" for t, p in sorted(pesos_otimos.items())])
        resultados_mensais.append({
            'Mes': data_rebal_original.strftime('%Y-%m'),
            'Data': data_rebal_original,
            'Portfolio': portfolio_str,
            'Retorno_Portfolio': retorno_portfolio,
            'Retorno_Ibov': retorno_ibov,
            'Alpha': alpha
        })
    
    # Salvar portfólios em CSV
    if salvar_csv and len(portfolios_mensais) > 0:
        df_portfolios = pd.DataFrame(portfolios_mensais)
        filename = f'portfolios_mensais_{modelo_tipo}_{data_inicio_teste.strftime("%Y-%m-%d")}_{data_fim_teste.strftime("%Y-%m-%d")}.csv'
        df_portfolios.to_csv(filename, index=False)
        print(f"\n✅ Portfólios salvos em: {filename}")
    
    # Resultados finais
    df_resultados = pd.DataFrame(resultados_mensais)
    if len(df_resultados) == 0:
        print("\n⚠ Nenhum resultado para processar")
        return None
    
    retorno_acumulado = (1 + df_resultados['Retorno_Portfolio']).prod() - 1
    retorno_ibov_acumulado = (1 + df_resultados['Retorno_Ibov']).prod() - 1
    volatilidade_portfolio = df_resultados['Retorno_Portfolio'].std() * np.sqrt(12) if len(df_resultados) > 1 else 0
    sharpe_ratio = (df_resultados['Retorno_Portfolio'].mean() * 12) / volatilidade_portfolio if volatilidade_portfolio > 0 else 0
    max_drawdown = calcular_max_drawdown(df_resultados['Retorno_Portfolio'].values)
    
    print(f"\n\n{'='*80}")
    print(f"RESULTADO FINAL - MODELO: {modelo_tipo}")
    print(f"{'='*80}")
    print(f"Retorno Acumulado do Portfólio: {retorno_acumulado*100:+.2f}%")
    print(f"Retorno Acumulado do Ibovespa: {retorno_ibov_acumulado*100:+.2f}%")
    print(f"Alpha Total: {(retorno_acumulado-retorno_ibov_acumulado)*100:+.2f}%")
    print(f"\nVolatilidade Anualizada: {volatilidade_portfolio*100:.2f}%")
    print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
    print(f"Max Drawdown: {max_drawdown*100:.2f}%")
    print(f"\nMeses Positivos: {(df_resultados['Retorno_Portfolio'] > 0).sum()}/{len(df_resultados)}")
    print(f"{'='*80}\n")

    # ============================================================================
    # PREVISÃO FORA DA AMOSTRA - MÊS SEGUINTE (CORRIGIDA)
    # ============================================================================

    # Data de referência = última data com dados disponível
    ultima_data_disponivel = min([df.index[-1] for df in dados_acoes.values()])
    data_referencia = ultima_data_disponivel

    print(f"\n{'='*80}")
    print(f"🔮 PREVISÃO FORA DA AMOSTRA")
    print(f"{'='*80}")
    print(f"📅 Última data com dados: {data_referencia.strftime('%Y-%m-%d')}")

    # Data limite para treino (deve ser 21 dias ANTES da predição)
    data_limite_treino_futuro = data_referencia - pd.Timedelta(days=target_lag)
    print(f"📅 Data limite treino: {data_limite_treino_futuro.strftime('%Y-%m-%d')} (= data_referencia - {target_lag}d)")
    print(f"{'='*80}\n")

    # Fazer predições para o próximo mês
    predicoes_futuro = {}
    ultimo_dia_treino = {}
    print(f"📊 PREDIÇÕES USANDO DADOS ATÉ {data_referencia.strftime('%Y-%m-%d')} (modelo {modelo_tipo}):\n")

    for ticker in dados_acoes.keys():
        # Usar dados ORIGINAIS até data_limite_treino_futuro
        df_original = dados_acoes[ticker]
        df_ate_limite = df_original[df_original.index <= data_limite_treino_futuro]
        
        if len(df_ate_limite) < 252:
            print(f" {ticker:12s} → ⚠ Dados insuficientes (apenas {len(df_ate_limite)} dias)")
            continue
        
        # Preparar features para este subconjunto
        df_prep_futuro, feature_cols_futuro = preparar_features_target(df_ate_limite, target_lag=target_lag)
        
        if len(df_prep_futuro) < 252:
            print(f" {ticker:12s} → ⚠ Features insuficientes após preparação")
            continue
        
        # Treinar modelo COM DADOS ATÉ data_limite_treino_futuro
        X_train = df_prep_futuro[feature_cols_futuro]
        y_train = df_prep_futuro['Target']
        
        if modelo_tipo == 'RF':
            model = RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                min_samples_split=20,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            )
        else:
            model = LinearRegression()
        
        model.fit(X_train, y_train)
        
        # ✅ CORREÇÃO: Buscar features para data_referencia
        # A data_referencia é a última data com dados
        try:
            linha_pred_preparada = df_prep_futuro[df_prep_futuro.index == data_referencia]
            
            if len(linha_pred_preparada) == 0:
                # Se data_referencia não está em df_prep_futuro, usar última linha
                # (que tem features defasadas corretas)
                linha_pred_preparada = df_prep_futuro.iloc[[-1]]
                data_real_predicao = df_prep_futuro.index[-1]
            else:
                data_real_predicao = data_referencia
        except Exception as e:
            print(f" {ticker:12s} → ⚠ Erro ao buscar features: {e}")
            continue
        
        # Fazer predição
        try:
            X_pred = linha_pred_preparada[feature_cols_futuro]
            predicao = model.predict(X_pred)[0]
            
            predicoes_futuro[ticker] = predicao
            ultimo_dia_treino[ticker] = df_prep_futuro.index[-1]
            
            print(f" {ticker:12s} → {predicao*100:+6.2f}% (treino até {df_prep_futuro.index[-1].strftime('%Y-%m-%d')}, predição usa {data_real_predicao.strftime('%Y-%m-%d')})")
        except Exception as e:
            print(f" {ticker:12s} → ⚠ Erro ao fazer predição: {e}")
            continue


    if len(predicoes_futuro) >= 3:
        print(f"\n✓ Predições realizadas: {len(predicoes_futuro)} ações\n")
        
        # Selecionar top 8
        predicoes_sorted_futuro = sorted(predicoes_futuro.items(), key=lambda x: x[1], reverse=True)
        top_acoes_futuro = predicoes_sorted_futuro[:min(8, len(predicoes_sorted_futuro))]
        
        print(f"🏆 TOP 8 AÇÕES COM MAIORES RETORNOS PREVISTOS:")
        for rank, (ticker, ret_pred) in enumerate(top_acoes_futuro, 1):
            ultimo_treino = ultimo_dia_treino[ticker]
            print(f" {rank}. {ticker:12s} → {ret_pred*100:+6.2f}% (treino até {ultimo_treino.strftime('%Y-%m-%d')})")
        
        # Calcular pesos otimizados
        tickers_top_futuro = [t for t, _ in top_acoes_futuro]
        
        # ✅ Para covariância, usar data_limite_treino_futuro
        print(f"\n📊 Calculando matriz de covariância...")
        cov_matrix_futuro = calcular_matriz_covariancia(
            tickers_top_futuro, 
            dados_acoes, 
            data_referencia,  # data_referencia já é a última data disponível
            janela=63, 
            target_lag=target_lag
        )
        
        retornos_esperados_futuro = {t: r for t, r in top_acoes_futuro}
        pesos_otimos_futuro = otimizar_markowitz(retornos_esperados_futuro, cov_matrix_futuro)
        
        print(f"\n💼 PORTFÓLIO RECOMENDADO (pesos otimizados):")
        retorno_esperado_portfolio = 0
        for ticker, peso in sorted(pesos_otimos_futuro.items(), key=lambda x: x[1], reverse=True):
            ret_esperado = retornos_esperados_futuro[ticker]
            retorno_esperado_portfolio += peso * ret_esperado
            print(f" {ticker:12s} → {peso*100:5.2f}% | Retorno esperado: {ret_esperado*100:+6.2f}%")
        
        print(f"\n📈 RETORNO ESPERADO DO PORTFÓLIO: {retorno_esperado_portfolio*100:+.2f}%")
        
        # Salvar previsão futura em CSV
        if salvar_csv:
            proximo_mes_str = (data_referencia + pd.DateOffset(months=1)).strftime('%Y-%m')
            portfolio_futuro = {
                'Mes': proximo_mes_str,
                'Data_Referencia': data_referencia.strftime('%Y-%m-%d'),
                'Data_Limite_Treino': data_limite_treino_futuro.strftime('%Y-%m-%d'),
                'Modelo': modelo_tipo,
                'Retorno_Esperado': retorno_esperado_portfolio
            }
            for ticker, peso in pesos_otimos_futuro.items():
                portfolio_futuro[ticker] = peso
            
            filename_futuro = f'previsao_{proximo_mes_str}_{modelo_tipo}_{data_referencia.strftime("%Y%m%d")}.csv'
            pd.DataFrame([portfolio_futuro]).to_csv(filename_futuro, index=False)
            print(f"\n✅ Previsão futura salva em: {filename_futuro}")

    else:
        print(f"\n⚠ Poucas predições disponíveis ({len(predicoes_futuro)}). Mínimo necessário: 3")
    
    print(f"Retorno Acumulado do Portfólio: {retorno_acumulado*100:+.2f}%")
    print(f"Retorno Acumulado do Ibovespa: {retorno_ibov_acumulado*100:+.2f}%")
    print(f"Alpha Total: {(retorno_acumulado-retorno_ibov_acumulado)*100:+.2f}%")

    return df_resultados

In [15]:
modelos = ['RF','media3m','menor_variancia','retorno_historico','retorno_ultimos_5m','retorno_ultimo_mes']


resultados = estrategia_portfolio_mensal(
        tickers=TICKERS,
        data_inicio_teste="2024-09-01",
        data_fim_teste="2025-10-01",
        n_acoes=8, modelo_tipo="RF", risco_adaptado=True
    )


ESTRATÉGIA DE PORTFÓLIO MENSAL - MODELO: RF
Período de teste: 2024-09-01 até 2025-10-01
Ações no universo: 8
Total de tickers: 12
Target lag: 21 dias

📥 Baixando dados históricos...
 • Período: 2019-09-01 até 2025-10-31
 → PETR4.SA... ✓ (1337 dias, até 2025-10-30)
 → VALE3.SA... ✓ (1337 dias, até 2025-10-30)
 → PRIO3.SA... ✓ (1337 dias, até 2025-10-30)
 → MGLU3.SA... ✓ (1337 dias, até 2025-10-30)
 → LREN3.SA... ✓ (1337 dias, até 2025-10-30)
 → ABEV3.SA... ✓ (1337 dias, até 2025-10-30)
 → WEGE3.SA... ✓ (1337 dias, até 2025-10-30)
 → ELET3.SA... ✓ (1337 dias, até 2025-10-30)
 → SUZB3.SA... ✓ (1337 dias, até 2025-10-30)
 → EMBR3.SA... ✓ (1337 dias, até 2025-10-30)
 → RDOR3.SA... ✓ (1017 dias, até 2025-10-30)
 → RAIL3.SA... ✓ (1337 dias, até 2025-10-30)

✓ Dados baixados para 12 ações
Preparando PETR4.SA...
Preparando VALE3.SA...
Preparando PRIO3.SA...
Preparando MGLU3.SA...
Preparando LREN3.SA...
Preparando ABEV3.SA...
Preparando WEGE3.SA...
Preparando ELET3.SA...
Preparando SUZB3.SA...


: 

In [17]:
def estrategia_portfolio_mensal_expandida(tickers, data_inicio_teste="2024-01-01",
                               data_fim_teste="2024-12-31", n_acoes=8,
                               risco_adaptado = True, salvar_csv=True, target_lag=21, k=15):

    target_lag = target_lag
    
    data_inicio_teste = pd.to_datetime(data_inicio_teste)
    data_fim_teste = pd.to_datetime(data_fim_teste)
    
    # Baixar dados até alguns dias APÓS data_fim_teste para ter margem
    data_inicio_download = data_inicio_teste - pd.DateOffset(years=5)
    data_fim_download = data_fim_teste + pd.DateOffset(days=30)
    
    print(f"\n{'='*80}")
    print(f"ESTRATÉGIA DE PORTFÓLIO MENSAL - MODELO:")
    print(f"{'='*80}")
    print(f"Período de teste: {data_inicio_teste.strftime('%Y-%m-%d')} até {data_fim_teste.strftime('%Y-%m-%d')}")
    print(f"Ações no universo: {n_acoes}")
    print(f"Total de tickers: {len(tickers)}")
    print(f"Target lag: {target_lag} dias")
    print(f"{'='*80}\n")
    
    print(f"📥 Baixando dados históricos...")
    print(f" • Período: {data_inicio_download.strftime('%Y-%m-%d')} até {data_fim_download.strftime('%Y-%m-%d')}")
    
    dados_acoes = {}
    for ticker in tickers:
        print(f" → {ticker}", end="... ")
        df = baixar_e_calcular_indicadores(
            ticker,
            start=data_inicio_download.strftime('%Y-%m-%d'),
            end=data_fim_download.strftime('%Y-%m-%d')
        )
        if df is not None and len(df) > 252:
            dados_acoes[ticker] = df
            print(f"✓ ({len(df)} dias, até {df.index[-1].strftime('%Y-%m-%d')})")
        else:
            print("✗ (sem dados suficientes)")
    
    print(f"\n✓ Dados baixados para {len(dados_acoes)} ações")
    
    if len(dados_acoes) < n_acoes:
        n_acoes = len(dados_acoes)
    

    dados_preparados = {}
    for ticker, df in dados_acoes.items():
        print(f"Preparando {ticker}...")
            
        df_prep, feature_cols = preparar_features_target(df, target_lag=21)
        if len(df_prep) > 0:
            dados_preparados[ticker] = (df_prep, feature_cols)





    
    print(f"✓ Features preparadas para {len(dados_preparados)} ações\n")
    
    # ✅ CORREÇÃO: Usar MS (month start) para primeira data, depois adicionar mês manualmente
    # Isso garante que as datas de rebalanceamento correspondam aos primeiros dias dos meses
    datas_rebalanceamento = pd.date_range(
        start=data_inicio_teste,
        end=data_fim_teste,
        freq='MS'
    )
    
    print(f"📅 Datas de rebalanceamento geradas: {len(datas_rebalanceamento)} meses")
    for i, d in enumerate(datas_rebalanceamento):
        print(f"   {i+1}. {d.strftime('%Y-%m-%d')}")
    print()

    modelos = ['menorvariancia','retornohistorico','retornoultimos5m','retornoultimomes']
    for modelo_tipo in modelos:
        resultados_mensais = []
        portfolios_mensais = []
        historico_predicoes = {}
        # Para cada mês
        for i, data_rebal_original in enumerate(datas_rebalanceamento):
            print(f"\n{'─'*80}")
            print(f"MÊS {i+1}: {data_rebal_original.strftime('%B/%Y').upper()}")
            print(f"{'─'*80}")
            
            # ✅ CORREÇÃO: Encontrar primeiro dia útil com dados a partir de data_rebal_original
            data_rebal = None
            offset_usado = 0
            
            for offset in range(10):
                data_teste = data_rebal_original + pd.Timedelta(days=offset)
                # Verificar se há dados disponíveis para esta data
                tem_dados = any(
                    len(df[df.index >= data_teste]) > 0
                    for df, _ in dados_preparados.values()
                )
                if tem_dados:
                    data_rebal = data_teste
                    offset_usado = offset
                    break
            
            if data_rebal is None:
                print(f"⚠ Sem data válida para rebalanceamento")
                continue
            
            if offset_usado > 0:
                print(f"📅 Data ajustada: {data_rebal.strftime('%Y-%m-%d')} (+{offset_usado} dias)")
            
            # ✅ CORREÇÃO: Data fim do mês = início do próximo mês - 1 dia
            if i < len(datas_rebalanceamento) - 1:
                # Próximo mês começa em datas_rebalanceamento[i+1]
                # Então fim deste mês é um dia antes
                data_fim_mes = datas_rebalanceamento[i+1] - pd.Timedelta(days=1)
            else:
                # Último mês: usar data_fim_teste ou última data disponível, o que for menor
                ultima_data_disponivel = min([df.index[-1] for df, _ in dados_preparados.values()])
                data_fim_mes = min(data_fim_teste, ultima_data_disponivel)
            
            print(f"📅 Período do mês: {data_rebal.strftime('%Y-%m-%d')} até {data_fim_mes.strftime('%Y-%m-%d')}")
            
            # Verificar se há dias úteis suficientes após data_rebal
            dias_disponiveis = []
            for df, _ in dados_preparados.values():
                dias_apos = len(df[(df.index > data_rebal) & (df.index <= data_fim_mes)])
                dias_disponiveis.append(dias_apos)
            
            if max(dias_disponiveis) < 2:
                print(f"⚠ Dias insuficientes após {data_rebal.strftime('%Y-%m-%d')} (máx: {max(dias_disponiveis)} dias)")
                continue

            predicoes = {}
            for ticker, (df, feature_cols) in dados_preparados.items():
                pred = treinar_e_prever(df, feature_cols, data_rebal, ticker, 
                                        modelo_tipo=modelo_tipo, target_lag=21, k=k)
                if pred is not None:
                    predicoes[ticker] = pred

            print(f"\n✓ Predições realizadas: {len(predicoes)} ações")
            
            if len(predicoes) < 3:
                print(f"⚠ Poucas predições, pulando")
                continue
            
            # Selecionar top N
            predicoes_sorted = sorted(predicoes.items(), key=lambda x: x[1], reverse=True)
            top_acoes = predicoes_sorted[:min(n_acoes, len(predicoes_sorted))]
            
            print(f"\n📊 TOP {len(top_acoes)} AÇÕES:")
            for rank, (ticker, ret_pred) in enumerate(top_acoes, 1):
                print(f" {rank}. {ticker:12s} → Retorno previsto: {ret_pred*100:+.2f}%")
            ###########################################################################
            # Covariância
            tickers_top = [t for t, _ in top_acoes]
            retornos_esperados = {t: r for t, r in top_acoes}

            if risco_adaptado:
                if i == 0:
                    # Primeiro mês: sem histórico ainda, usar função original
                    cov_matrix = calcular_matriz_covariancia(
                        tickers_top, dados_acoes, data_rebal, 
                        janela=63, target_lag=target_lag
                    )
                else:
                    # A partir do segundo mês: usar função ajustada
                    cov_matrix = calcular_matriz_covariancia_ajustada(
                        tickers_top, dados_acoes, historico_predicoes,
                        data_rebal, janela=63, target_lag=target_lag
                    )
            else:
                cov_matrix = calcular_matriz_covariancia(
                    tickers_top, dados_acoes, data_rebal, 
                    janela=63, target_lag=target_lag
                )
            
            # Resto continua igual (otimizar_markowitz, etc.)
            pesos_otimos = otimizar_markowitz(retornos_esperados, cov_matrix)

            
            print(f"\n🎯 PESOS OTIMIZADOS:")
            for ticker, peso in sorted(pesos_otimos.items(), key=lambda x: x[1], reverse=True):
                print(f" {ticker:12s} → {peso*100:5.2f}%")
            #############################################################################################
            # Salvar portfólio do mês
            portfolio_mes = {
                'Mes': data_rebal_original.strftime('%Y-%m'),
                'Data': data_rebal_original.strftime('%Y-%m-%d'),
                'Modelo': modelo_tipo
            }
            for ticker, peso in pesos_otimos.items():
                portfolio_mes[ticker] = peso
            portfolios_mensais.append(portfolio_mes)
            
            # Calcular retornos reais
            print(f"\n🔍 Calculando retornos reais DE {data_rebal.strftime('%Y-%m-%d')} ATÉ {data_fim_mes.strftime('%Y-%m-%d')}")
            
            retornos_reais = {}
            print(f"\n📈 RETORNOS REAIS:")
            for ticker in pesos_otimos.keys():
                df_acao = dados_acoes[ticker]
                df_periodo = df_acao[(df_acao.index > data_rebal) & (df_acao.index <= data_fim_mes)]
                
                if len(df_periodo) >= 2:
                    preco_inicio = df_periodo['Close'].iloc[0]
                    preco_fim = df_periodo['Close'].iloc[-1]
                    retorno_real = (preco_fim - preco_inicio) / preco_inicio
                    retornos_reais[ticker] = retorno_real
                    
                    ret_previsto = retornos_esperados[ticker]
                    erro = abs(retorno_real - ret_previsto)
                    print(f" {ticker:12s}: Real={retorno_real*100:+6.2f}% | Previsto={ret_previsto*100:+6.2f}% | Erro={erro*100:6.2f}%")
                else:
                    retornos_reais[ticker] = 0
                # Se fizemos previsão E temos retorno real para esta ação
                if ticker in predicoes and ticker in retornos_reais:
                    
                    # Criar entrada se não existe
                    if ticker not in historico_predicoes:
                        historico_predicoes[ticker] = {
                            'previsto': [],
                            'real': []
                        }
                    
                    # Adicionar este mês
                    historico_predicoes[ticker]['previsto'].append(predicoes[ticker])
                    historico_predicoes[ticker]['real'].append(retornos_reais[ticker])
            
            # Retorno do portfólio
            retorno_portfolio = sum(retornos_reais.get(t, 0) * peso for t, peso in pesos_otimos.items())

            # Benchmark
            ibov = yf.download("^BVSP", start=data_rebal, end=data_fim_mes, progress=False)
            if not ibov.empty and len(ibov) >= 2:
                if isinstance(ibov['Close'], pd.Series):
                    preco_ibov_inicio = ibov['Close'].iloc[0]
                    preco_ibov_fim = ibov['Close'].iloc[-1]
                else:
                    preco_ibov_inicio = ibov['Close'].iloc[0].values[0]
                    preco_ibov_fim = ibov['Close'].iloc[-1].values[0]
                retorno_ibov = (preco_ibov_fim - preco_ibov_inicio) / preco_ibov_inicio
            else:
                retorno_ibov = 0
            
            alpha = retorno_portfolio - retorno_ibov
            
            print(f"\n💰 RESULTADO DO MÊS:")
            print(f" Retorno do Portfólio: {retorno_portfolio*100:+.2f}%")
            print(f" Retorno do Ibovespa: {retorno_ibov*100:+.2f}%")
            print(f" Alpha: {alpha*100:+.2f}%")
            
            portfolio_str = ', '.join([f"{t} ({p*100:.1f}%)" for t, p in sorted(pesos_otimos.items())])
            resultados_mensais.append({
                'Mes': data_rebal_original.strftime('%Y-%m'),
                'Data': data_rebal_original,
                'Portfolio': portfolio_str,
                'Retorno_Portfolio': retorno_portfolio,
                'Retorno_Ibov': retorno_ibov,
                'Alpha': alpha
            })
        
        # Salvar portfólios em CSV
        if salvar_csv and len(portfolios_mensais) > 0:
            df_portfolios = pd.DataFrame(portfolios_mensais)
            filename = f'portfolios_mensais_{modelo_tipo}_{data_inicio_teste.strftime("%Y-%m-%d")}_{data_fim_teste.strftime("%Y-%m-%d")}.csv'
            df_portfolios.to_csv(filename, index=False)
            print(f"\n✅ Portfólios salvos em: {filename}")
        
        # Resultados finais
        df_resultados = pd.DataFrame(resultados_mensais)
        if len(df_resultados) == 0:
            print("\n⚠ Nenhum resultado para processar")
            return None
        
        retorno_acumulado = (1 + df_resultados['Retorno_Portfolio']).prod() - 1
        retorno_ibov_acumulado = (1 + df_resultados['Retorno_Ibov']).prod() - 1
        volatilidade_portfolio = df_resultados['Retorno_Portfolio'].std() * np.sqrt(12) if len(df_resultados) > 1 else 0
        sharpe_ratio = (df_resultados['Retorno_Portfolio'].mean() * 12) / volatilidade_portfolio if volatilidade_portfolio > 0 else 0
        max_drawdown = calcular_max_drawdown(df_resultados['Retorno_Portfolio'].values)
        
        print(f"\n\n{'='*80}")
        print(f"RESULTADO FINAL - MODELO: {modelo_tipo}")
        print(f"{'='*80}")
        print(f"Retorno Acumulado do Portfólio: {retorno_acumulado*100:+.2f}%")
        print(f"Retorno Acumulado do Ibovespa: {retorno_ibov_acumulado*100:+.2f}%")
        print(f"Alpha Total: {(retorno_acumulado-retorno_ibov_acumulado)*100:+.2f}%")
        print(f"\nVolatilidade Anualizada: {volatilidade_portfolio*100:.2f}%")
        print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
        print(f"Max Drawdown: {max_drawdown*100:.2f}%")
        print(f"\nMeses Positivos: {(df_resultados['Retorno_Portfolio'] > 0).sum()}/{len(df_resultados)}")
        print(f"{'='*80}\n")

        # ============================================================================
        # PREVISÃO FORA DA AMOSTRA - MÊS SEGUINTE (CORRIGIDA)
        # ============================================================================

        # Data de referência = última data com dados disponível
        ultima_data_disponivel = min([df.index[-1] for df in dados_acoes.values()])
        data_referencia = ultima_data_disponivel

        print(f"\n{'='*80}")
        print(f"🔮 PREVISÃO FORA DA AMOSTRA")
        print(f"{'='*80}")
        print(f"📅 Última data com dados: {data_referencia.strftime('%Y-%m-%d')}")

        # Data limite para treino (deve ser 21 dias ANTES da predição)
        data_limite_treino_futuro = data_referencia - pd.Timedelta(days=target_lag)
        print(f"📅 Data limite treino: {data_limite_treino_futuro.strftime('%Y-%m-%d')} (= data_referencia - {target_lag}d)")
        print(f"{'='*80}\n")

        # Fazer predições para o próximo mês
        predicoes_futuro = {}
        ultimo_dia_treino = {}
        print(f"📊 PREDIÇÕES USANDO DADOS ATÉ {data_referencia.strftime('%Y-%m-%d')} (modelo {modelo_tipo}):\n")

        for ticker in dados_acoes.keys():
            # Usar dados ORIGINAIS até data_limite_treino_futuro
            df_original = dados_acoes[ticker]
            df_ate_limite = df_original[df_original.index <= data_limite_treino_futuro]
            
            if len(df_ate_limite) < 252:
                print(f" {ticker:12s} → ⚠ Dados insuficientes (apenas {len(df_ate_limite)} dias)")
                continue
            
            # Preparar features para este subconjunto
            df_prep_futuro, feature_cols_futuro = preparar_features_target(df_ate_limite, target_lag=target_lag)
            
            if len(df_prep_futuro) < 252:
                print(f" {ticker:12s} → ⚠ Features insuficientes após preparação")
                continue
            
            # Treinar modelo COM DADOS ATÉ data_limite_treino_futuro
            X_train = df_prep_futuro[feature_cols_futuro]
            y_train = df_prep_futuro['Target']
            
            if modelo_tipo == 'RF':
                model = RandomForestRegressor(
                    n_estimators=100,
                    max_depth=10,
                    min_samples_split=20,
                    min_samples_leaf=10,
                    random_state=42,
                    n_jobs=-1
                )
            else:
                model = LinearRegression()
            
            model.fit(X_train, y_train)
            
            # ✅ CORREÇÃO: Buscar features para data_referencia
            # A data_referencia é a última data com dados
            try:
                linha_pred_preparada = df_prep_futuro[df_prep_futuro.index == data_referencia]
                
                if len(linha_pred_preparada) == 0:
                    # Se data_referencia não está em df_prep_futuro, usar última linha
                    # (que tem features defasadas corretas)
                    linha_pred_preparada = df_prep_futuro.iloc[[-1]]
                    data_real_predicao = df_prep_futuro.index[-1]
                else:
                    data_real_predicao = data_referencia
            except Exception as e:
                print(f" {ticker:12s} → ⚠ Erro ao buscar features: {e}")
                continue
            
            # Fazer predição
            try:
                X_pred = linha_pred_preparada[feature_cols_futuro]
                predicao = model.predict(X_pred)[0]
                
                predicoes_futuro[ticker] = predicao
                ultimo_dia_treino[ticker] = df_prep_futuro.index[-1]
                
                print(f" {ticker:12s} → {predicao*100:+6.2f}% (treino até {df_prep_futuro.index[-1].strftime('%Y-%m-%d')}, predição usa {data_real_predicao.strftime('%Y-%m-%d')})")
            except Exception as e:
                print(f" {ticker:12s} → ⚠ Erro ao fazer predição: {e}")
                continue


        if len(predicoes_futuro) >= 3:
            print(f"\n✓ Predições realizadas: {len(predicoes_futuro)} ações\n")
            
            # Selecionar top 8
            predicoes_sorted_futuro = sorted(predicoes_futuro.items(), key=lambda x: x[1], reverse=True)
            top_acoes_futuro = predicoes_sorted_futuro[:min(8, len(predicoes_sorted_futuro))]
            
            print(f"🏆 TOP 8 AÇÕES COM MAIORES RETORNOS PREVISTOS:")
            for rank, (ticker, ret_pred) in enumerate(top_acoes_futuro, 1):
                ultimo_treino = ultimo_dia_treino[ticker]
                print(f" {rank}. {ticker:12s} → {ret_pred*100:+6.2f}% (treino até {ultimo_treino.strftime('%Y-%m-%d')})")
            
            # Calcular pesos otimizados
            tickers_top_futuro = [t for t, _ in top_acoes_futuro]
            
            # ✅ Para covariância, usar data_limite_treino_futuro
            print(f"\n📊 Calculando matriz de covariância...")
            cov_matrix_futuro = calcular_matriz_covariancia(
                tickers_top_futuro, 
                dados_acoes, 
                data_referencia,  # data_referencia já é a última data disponível
                janela=63, 
                target_lag=target_lag
            )
            
            retornos_esperados_futuro = {t: r for t, r in top_acoes_futuro}
            pesos_otimos_futuro = otimizar_markowitz(retornos_esperados_futuro, cov_matrix_futuro)
            
            print(f"\n💼 PORTFÓLIO RECOMENDADO (pesos otimizados):")
            retorno_esperado_portfolio = 0
            for ticker, peso in sorted(pesos_otimos_futuro.items(), key=lambda x: x[1], reverse=True):
                ret_esperado = retornos_esperados_futuro[ticker]
                retorno_esperado_portfolio += peso * ret_esperado
                print(f" {ticker:12s} → {peso*100:5.2f}% | Retorno esperado: {ret_esperado*100:+6.2f}%")
            
            print(f"\n📈 RETORNO ESPERADO DO PORTFÓLIO: {retorno_esperado_portfolio*100:+.2f}%")
            
            # Salvar previsão futura em CSV
            if salvar_csv:
                proximo_mes_str = (data_referencia + pd.DateOffset(months=1)).strftime('%Y-%m')
                portfolio_futuro = {
                    'Mes': proximo_mes_str,
                    'Data_Referencia': data_referencia.strftime('%Y-%m-%d'),
                    'Data_Limite_Treino': data_limite_treino_futuro.strftime('%Y-%m-%d'),
                    'Modelo': modelo_tipo,
                    'Retorno_Esperado': retorno_esperado_portfolio
                }
                for ticker, peso in pesos_otimos_futuro.items():
                    portfolio_futuro[ticker] = peso
                
                filename_futuro = f'previsao_{proximo_mes_str}_{modelo_tipo}_{data_referencia.strftime("%Y%m%d")}.csv'
                pd.DataFrame([portfolio_futuro]).to_csv(filename_futuro, index=False)
                print(f"\n✅ Previsão futura salva em: {filename_futuro}")

        else:
            print(f"\n⚠ Poucas predições disponíveis ({len(predicoes_futuro)}). Mínimo necessário: 3")
        
        print(f"Retorno Acumulado do Portfólio: {retorno_acumulado*100:+.2f}%")
        print(f"Retorno Acumulado do Ibovespa: {retorno_ibov_acumulado*100:+.2f}%")
        print(f"Alpha Total: {(retorno_acumulado-retorno_ibov_acumulado)*100:+.2f}%")

    return df_resultados

In [18]:
resultados = estrategia_portfolio_mensal_expandida(TICKERS_EXPANDIDA,data_inicio_teste="2023-01-01",
                               data_fim_teste="2025-11-01")


ESTRATÉGIA DE PORTFÓLIO MENSAL - MODELO:
Período de teste: 2023-01-01 até 2025-11-01
Ações no universo: 8
Total de tickers: 92
Target lag: 21 dias

📥 Baixando dados históricos...
 • Período: 2018-01-01 até 2025-12-01
 → ITUB4.SA... ✓ (1769 dias, até 2025-11-28)
 → BBDC4.SA... ✓ (1769 dias, até 2025-11-28)
 → BBAS3.SA... ✓ (1769 dias, até 2025-11-28)
 → SANB11.SA... ✓ (1769 dias, até 2025-11-28)
 → BPAC11.SA... ✓ (1769 dias, até 2025-11-28)
 → CXSE3.SA... ✓ (948 dias, até 2025-11-28)
 → BRAP4.SA... ✓ (1769 dias, até 2025-11-28)
 → BRSR6.SA... ✓ (1769 dias, até 2025-11-28)
 → CRFB3.SA... 


1 Failed download:
['CRFB3.SA']: YFTzMissingError('possibly delisted; no timezone found')


✗ (sem dados suficientes)
 → PSSA3.SA... ✓ (1769 dias, até 2025-11-28)
 → PINE4.SA... ✓ (1769 dias, até 2025-11-28)
 → PETR4.SA... ✓ (1769 dias, até 2025-11-28)
 → PRIO3.SA... ✓ (1769 dias, até 2025-11-28)
 → OIBR4.SA... ✓ (1769 dias, até 2025-11-28)
 → ELET3.SA... ✓ (1769 dias, até 2025-11-28)
 → CMIG4.SA... ✓ (1769 dias, até 2025-11-28)
 → CPFE3.SA... ✓ (1769 dias, até 2025-11-28)
 → EGIE3.SA... ✓ (1769 dias, até 2025-11-28)
 → ENGI11.SA... ✓ (1769 dias, até 2025-11-28)
 → GEMA3.SA... 


1 Failed download:
['GEMA3.SA']: YFTzMissingError('possibly delisted; no timezone found')


✗ (sem dados suficientes)
 → LIGHT3.SA... 

...

✗ (sem dados suficientes)
 → SGUP3.SA... 


1 Failed download:
['SGUP3.SA']: YFTzMissingError('possibly delisted; no timezone found')


✗ (sem dados suficientes)
 → B3SA3.SA... ✓ (1769 dias, até 2025-11-28)
 → MOVI3.SA... ✓ (1769 dias, até 2025-11-28)
 → RBRR3.SA... 


1 Failed download:
['RBRR3.SA']: YFTzMissingError('possibly delisted; no timezone found')


✗ (sem dados suficientes)
 → RDOR3.SA... ✓ (1037 dias, até 2025-11-28)
 → AGRO3.SA... ✓ (1769 dias, até 2025-11-28)
 → AERI3.SA... ✓ (1059 dias, até 2025-11-28)
 → POSI3.SA... ✓ (1769 dias, até 2025-11-28)

✓ Dados baixados para 65 ações
Preparando ITUB4.SA...
Preparando BBDC4.SA...
Preparando BBAS3.SA...
Preparando SANB11.SA...
Preparando BPAC11.SA...
Preparando CXSE3.SA...
Preparando BRAP4.SA...
Preparando BRSR6.SA...
Preparando PSSA3.SA...
Preparando PINE4.SA...
Preparando PETR4.SA...
Preparando PRIO3.SA...
Preparando OIBR4.SA...
Preparando ELET3.SA...
Preparando CMIG4.SA...
Preparando CPFE3.SA...
Preparando EGIE3.SA...
Preparando ENGI11.SA...
Preparando EQTL3.SA...
Preparando VALE3.SA...
Preparando CSNA3.SA...
Preparando USIM5.SA...
Preparando GGBR4.SA...
Preparando MGLU3.SA...
Preparando LREN3.SA...
Preparando ABEV3.SA...
Preparando RENT3.SA...
Preparando MOVI3.SA...
Preparando PCAR3.SA...
Preparando TRIS3.SA...
Preparando WEGE3.SA...
Preparando MSFT34.SA...
Preparando HYPE3.SA...